<a href="https://colab.research.google.com/github/KK-Dev0/Stuck_Doctor/blob/main/Stuck_Doctor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!sudo apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [16]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [17]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [18]:
!ollama pull gemma4

In [19]:
!pip install ollama

In [20]:
from ollama import chat

response = chat(model='gemma4', messages=[
    {'role': 'user', 'content': 'Break "prepare for exam" into 5 small steps'}
])
print(response.message.content)

Here are 5 small, actionable steps to break down "prepare for exam":

1. **Assess and Plan (The Map):** Identify exactly what topics will be covered, find all necessary study materials (textbooks, notes, practice tests), and create a realistic study schedule. *(Knowing what you need to know.)*

2. **Active Learning (The Immersion):** Don't just re-read. Actively engage with the material. Use flashcards, summarize concepts in your own words, and watch explanatory videos for difficult subjects. *(Learning the content deeply.)*

3. **Practice Application (The Drill):** Work through past exams, sample questions, or problem sets. This phase is crucial for understanding *how* the information is tested, not just *what* the information is. *(Testing your knowledge and skills.)*

4. **Review and Refine (The Polish):** Use your practice test results to identify your weakest areas (e.g., Chapter 4, certain type of math problem). Go back to the material specifically to fill those knowledge gaps. *

In [26]:
import subprocess, time, requests

def ensure_ollama():
    try:
        requests.get("http://127.0.0.1:11434", timeout=2)
        return
    except Exception:
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"),
                         stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(20):
            time.sleep(1)
            try:
                requests.get("http://127.0.0.1:11434", timeout=2)
                return
            except Exception:
                pass

ensure_ollama()

In [ ]:
import json, datetime
from urllib.parse import quote
from zoneinfo import ZoneInfo
import gradio as gr
from ollama import chat

gr.close_all()  # stop the old version

GUIDE_ME = "I'm not sure, guide me"
TIMES = ["In 1 hour", "Tonight (7 PM)", "Tomorrow morning (10 AM)", "This Saturday (11 AM)"]
WHEN = [GUIDE_ME] + TIMES
REASONS = ["Scared of bad news", "I hate calling to book", "It's probably nothing",
           "Don't know which doctor to see", "Worried about cost or time", "It feels awkward or embarrassing"]
LANGUAGES = ["English", "Hinglish", "Bengali"]

PROMPT = f"""You are Stuck Doctor, a calm, kind helper for people in India who keep putting off going to the doctor.
You do NOT diagnose or give medical advice. You only help them take the next step toward care.
Reply ONLY with JSON using exactly these keys:
"urgent": true if anything sounds urgent or serious (chest pain, trouble breathing, heavy bleeding, fainting, sudden severe pain), otherwise false
"diagnosis": one kind sentence naming the type of stuck they are in
"first_move": one specific action they can do in 2 minutes
"doctor_type": the kind of doctor to search for, in 1-3 words (use "general physician" if unsure)
"message": a short WhatsApp message to a clinic asking to book an appointment, written in the requested language
"questions": a list of 3 short questions the patient should ask the doctor during the visit
"while_you_wait": a list of 2 gentle, general comfort tips that involve no medicines, doses, home remedies or putting anything in the body
"see_sooner_if": a list of 2-3 warning signs that mean they should get care urgently
"suggested_when": exactly one of {TIMES}. Pick the soonest realistic time for the first move. Prefer today unless it is late at night. Never suggest waiting because of fear or worry, since delay makes that worse. Always pick "In 1 hour" if urgent is true.
"why_this_time": one short sentence explaining why that time is a good choice
"push": one encouraging line
Do not mention insurance unless the user does."""

def pick_time(when):
    now = datetime.datetime.now(ZoneInfo("Asia/Kolkata"))
    if when == "In 1 hour":
        return now + datetime.timedelta(hours=1)
    if when == "Tonight (7 PM)":
        t = now.replace(hour=19, minute=0, second=0, microsecond=0)
        return t if t > now else t + datetime.timedelta(days=1)
    if when == "Tomorrow morning (10 AM)":
        return (now + datetime.timedelta(days=1)).replace(hour=10, minute=0, second=0, microsecond=0)
    days = (5 - now.weekday()) % 7 or 7
    return (now + datetime.timedelta(days=days)).replace(hour=11, minute=0, second=0, microsecond=0)

def button(url, text, color):
    return (f'<a href="{url}" target="_blank" style="display:inline-block;margin:6px;padding:12px 18px;'
            f'background:{color};color:white;border-radius:10px;text-decoration:none;font-weight:600;">{text}</a>')

def bullets(items):
    return "\n".join(f"- {i}" for i in items) + "\n\n"

def stuck_doctor(task, reason, when, language, details):
    ensure_ollama()
    user = f"What I'm putting off: {task}\nWhy I'm stuck: {reason}\nLanguage for message: {language}\nAnything else: {details}"
    r = chat(model='gemma4', format='json', messages=[
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': user}
    ])
    try:
        d = json.loads(r.message.content)
    except json.JSONDecodeError:
        return "Stuck Doctor got confused 😵 Please click the button again.", ""

    # decide the time
    if when == GUIDE_ME:
        chosen = d.get("suggested_when")
        if d.get("urgent"):
            chosen = "In 1 hour"
        if chosen not in TIMES:
            chosen = "Tomorrow morning (10 AM)"
    else:
        chosen = when

    text = ""
    if d.get("urgent"):
        text += "### ⚠️ This sounds like it needs attention soon. Please see a doctor or get medical help today.\n\n"
    text += f"**Diagnosis:** {d.get('diagnosis','')}\n\n"
    text += f"**Your 2-minute first move:** {d.get('first_move','')}\n\n"
    if when == GUIDE_ME:
        text += f"**Best time to do it:** {chosen}. {d.get('why_this_time','')}\n\n"
    text += f"**Message to send:**\n> {d.get('message','')}\n\n"
    text += "**Questions to ask the doctor:**\n" + bullets(d.get('questions', []))
    text += "**While you wait:**\n" + bullets(d.get('while_you_wait', []))
    text += "**See a doctor sooner if:**\n" + bullets(d.get('see_sooner_if', []))
    text += f"**Little push:** {d.get('push','')}"

    start = pick_time(chosen)
    end = start + datetime.timedelta(minutes=15)
    fmt = "%Y%m%dT%H%M%S"
    whatsapp = "https://wa.me/?text=" + quote(d.get('message', ''))
    maps = "https://www.google.com/maps/search/?api=1&query=" + quote(d.get('doctor_type', 'doctor') + " near me")
    calendar = ("https://calendar.google.com/calendar/render?action=TEMPLATE"
                f"&text={quote('Stuck Doctor: book a doctor for ' + task)}&details={quote(d.get('first_move',''))}"
                f"&dates={start.strftime(fmt)}/{end.strftime(fmt)}&ctz=Asia/Kolkata")

    buttons = (button(whatsapp, "💬 Send on WhatsApp", "#25D366")
               + button(maps, f"📍 Find a {d.get('doctor_type','doctor')}", "#4285F4")
               + button(calendar, f"📅 Remind me to book: {chosen}", "#EA4335"))
    return text, buttons

with gr.Blocks(title="Stuck Doctor") as app:
    gr.Markdown("# 🩺 Stuck Doctor\nFor when you keep putting off the doctor. Tell me why you're stuck, and I'll get you moving. *(Not medical advice.)*")
    with gr.Row():
        with gr.Column():
            task = gr.Textbox(label="What are you putting off?", placeholder="e.g. seeing a doctor about my ear ache")
            reason = gr.Dropdown(REASONS, label="Why are you stuck?", value=REASONS[0])
            when = gr.Dropdown(WHEN, label="When will you take the first step?", value=GUIDE_ME)
            language = gr.Radio(LANGUAGES, label="Message language", value="English")
            details = gr.Textbox(label="Anything else? (optional)")
            go = gr.Button("Get my prescription 💊", variant="primary")
        with gr.Column():
            result = gr.Markdown()
            actions = gr.HTML()
    go.click(stuck_doctor, inputs=[task, reason, when, language, details], outputs=[result, actions])

app.launch(share=True, debug=True)

Closing server running on port: 7860
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8093bb6a7094aec231.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [22]:
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama list

NAME             ID              SIZE      MODIFIED       
gemma4:latest    dc35e8d9c606    6.6 GB    51 seconds ago    
